# ChestX-ray14 — Train DenseNet-121 (14 classes) on Kaggle

**Self-contained.** Attach the [NIH Chest X-rays](https://www.kaggle.com/datasets/nih-chest-xrays/data) dataset and *Run All*. No other files needed.

Pipeline: **preprocess (clean + undersample No-Finding) → train → evaluate (per-class AUC + healthy verdict)**.

We train on the **14 pathologies**. A healthy scan is the *absence of all 14* — we report a derived **`No Finding score = 1 − max(probabilities)`** so the model can still say *"this looks healthy."*

> Research/educational only — not for clinical use.

## 1. Config & paths

In [ ]:
import os, glob, numpy as np, pandas as pd, torch

# --- locate the attached NIH dataset (handles the usual Kaggle layouts) ---
def find_archive():
    for root, _, files in os.walk('/kaggle/input'):
        if 'Data_Entry_2017.csv' in files:
            return root
    raise FileNotFoundError('Data_Entry_2017.csv not found under /kaggle/input — attach the NIH dataset.')

ARCHIVE   = find_archive()
IMAGE_ROOT = ARCHIVE                       # images_XXX/images/ live under here
OUT        = '/kaggle/working'
CKPT_DIR   = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('archive :', ARCHIVE)
print('device  :', DEVICE)

LABELS = ['Atelectasis','Cardiomegaly','Consolidation','Edema','Effusion','Emphysema',
          'Fibrosis','Hernia','Infiltration','Mass','Nodule','Pleural_Thickening',
          'Pneumonia','Pneumothorax']
NUM_CLASSES = len(LABELS)

# hyperparameters
BATCH_SIZE, NUM_EPOCHS, NUM_WORKERS, IMG_SIZE = 32, 15, 4, 224
LR_BACKBONE, LR_HEAD, WEIGHT_DECAY = 2e-5, 2e-4, 1e-4
NF_RATIO, POS_CAP, VAL_FRAC, SEED = 0.5, 10.0, 0.10, 42

## 2. Preprocess — clean, official split, undersample No-Finding

Drops impossible ages, encodes gender (M=0/F=1) and normalizes age (/100), expands the 14 binary labels, splits with the **official patient-wise NIH lists**, and **undersamples healthy images in train only** so pathologies get a stronger signal. Val/test keep their natural distribution for honest evaluation.

In [ ]:
df = pd.read_csv(os.path.join(ARCHIVE, 'Data_Entry_2017.csv'))
df = df[df['Patient Age'] <= 100].copy()
df['Patient Gender'] = (df['Patient Gender'] == 'F').astype('float32')
df['Patient Age'] = df['Patient Age'].astype('float32') / 100.0
for L in LABELS:
    df[L] = df['Finding Labels'].fillna('').str.contains(L, regex=False).astype('float32')
df = df[['Image Index','Patient ID','Patient Age','Patient Gender'] + LABELS]

tv = set(open(os.path.join(ARCHIVE,'train_val_list.txt')).read().split())
te = set(open(os.path.join(ARCHIVE,'test_list.txt')).read().split())
trainval, test = df[df['Image Index'].isin(tv)].copy(), df[df['Image Index'].isin(te)].copy()

rng = np.random.default_rng(SEED)
pats = trainval['Patient ID'].unique(); rng.shuffle(pats)
val_pats = set(pats[:int(len(pats)*VAL_FRAC)])
val   = trainval[trainval['Patient ID'].isin(val_pats)].copy()
train = trainval[~trainval['Patient ID'].isin(val_pats)].copy()

# undersample No-Finding in train
has = train[LABELS].sum(axis=1) > 0
pos, neg = train[has], train[~has]
keep = int(len(pos)*NF_RATIO)
if keep < len(neg):
    neg = neg.sample(n=keep, random_state=SEED)
train = pd.concat([pos, neg]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

for name, d in [('train',train),('val',val),('test',test)]:
    d.drop(columns=['Patient ID']).to_csv(f'{OUT}/{name}.csv', index=False)

healthy = lambda d: (d[LABELS].sum(axis=1)==0).mean()
print(f'train {len(train):>6,} (healthy {healthy(train):.0%}) | '
      f'val {len(val):>6,} (healthy {healthy(val):.0%}) | '
      f'test {len(test):>6,} (healthy {healthy(test):.0%})')

## 3. Dataset, transforms & model

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn

MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

def tfm(train):
    if train:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.RandomHorizontalFlip(),
            transforms.RandomAffine(7, translate=(0.05,0.05), scale=(0.95,1.05)),
            transforms.ColorJitter(0.1,0.1), transforms.ToTensor(),
            transforms.Normalize(MEAN,STD), transforms.RandomErasing(p=0.25, scale=(0.02,0.08))])
    return transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),
                               transforms.ToTensor(), transforms.Normalize(MEAN,STD)])

print('indexing images...')
IMG_INDEX = {f: os.path.join(d,f) for d,_,fs in os.walk(IMAGE_ROOT) for f in fs if f.endswith('.png')}
print('found', len(IMG_INDEX), 'images')

class ChestXray(Dataset):
    def __init__(self, csv, train):
        self.df = pd.read_csv(csv); self.t = tfm(train)
        self.y = self.df[LABELS].values.astype('float32')
        self.age = self.df['Patient Age'].values.astype('float32')
        self.sex = self.df['Patient Gender'].values.astype('float32')
        self.f = self.df['Image Index'].values
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        img = self.t(Image.open(IMG_INDEX[self.f[i]]).convert('RGB'))
        return img, torch.tensor([self.age[i]]), torch.tensor([self.sex[i]]), torch.from_numpy(self.y[i])

class ChestXrayModel(nn.Module):
    def __init__(self, n=NUM_CLASSES, pretrained=True):
        super().__init__()
        try:
            w = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
            d = models.densenet121(weights=w)
        except AttributeError:
            d = models.densenet121(pretrained=pretrained)
        self.features = d.features
        self.gap = nn.AdaptiveAvgPool2d((1,1))
        self.clinical_branch = nn.Sequential(nn.Linear(2,16), nn.ReLU(True), nn.Dropout(0.3))
        self.classifier = nn.Sequential(nn.Linear(1040,512), nn.ReLU(True), nn.Dropout(0.5), nn.Linear(512,n))
    def forward(self, img, age, sex):
        x = torch.flatten(self.gap(torch.relu(self.features(img))), 1)
        c = self.clinical_branch(torch.cat([age, sex], 1))
        return self.classifier(torch.cat([x, c], 1))

## 4. Train

In [ ]:
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm

kw = dict(num_workers=NUM_WORKERS, pin_memory=True,
          persistent_workers=NUM_WORKERS>0, prefetch_factor=2 if NUM_WORKERS else None)
train_loader = DataLoader(ChestXray(f'{OUT}/train.csv', True),  batch_size=BATCH_SIZE,   shuffle=True,  **kw)
val_loader   = DataLoader(ChestXray(f'{OUT}/val.csv',   False), batch_size=BATCH_SIZE*2, shuffle=False, **kw)

tr = pd.read_csv(f'{OUT}/train.csv')
pos_w = torch.tensor([min((len(tr)-tr[L].sum())/max(tr[L].sum(),1), POS_CAP) for L in LABELS],
                     dtype=torch.float32).to(DEVICE)

model = ChestXrayModel().to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
scaler = GradScaler()
opt = torch.optim.AdamW([
    {'params': model.features.parameters(), 'lr': LR_BACKBONE},
    {'params': list(model.clinical_branch.parameters())+list(model.classifier.parameters()), 'lr': LR_HEAD},
], weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR_BACKBONE,LR_HEAD],
          epochs=NUM_EPOCHS, steps_per_epoch=len(train_loader), pct_start=0.2)

def class_auc(y, p):
    a = [roc_auc_score(y[:,i], p[:,i]) if 0 < y[:,i].sum() < len(y) else np.nan for i in range(y.shape[1])]
    return a, float(np.nanmean(a))

@torch.no_grad()
def evaluate(loader):
    model.eval(); P, Y = [], []
    for img, age, sex, y in tqdm(loader, desc='val', leave=False):
        with autocast():
            logits = model(img.to(DEVICE), age.to(DEVICE), sex.to(DEVICE))
        P.append(torch.sigmoid(logits.float()).cpu().numpy()); Y.append(y.numpy())
    P, Y = np.concatenate(P), np.concatenate(Y)
    return Y, P, class_auc(Y, P)[1]

best = 0.0
for ep in range(NUM_EPOCHS):
    model.train(); run = 0.0
    for img, age, sex, y in tqdm(train_loader, desc=f'epoch {ep+1}/{NUM_EPOCHS}'):
        img, age, sex, y = img.to(DEVICE), age.to(DEVICE), sex.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        with autocast():
            loss = criterion(model(img, age, sex), y)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
    Y, P, auc = evaluate(val_loader)
    print(f'epoch {ep+1:02d} | train_loss {run/len(train_loader):.4f} | val mean_AUC {auc:.4f}')
    if auc > best:
        best = auc
        torch.save({'model_state_dict': model.state_dict(), 'val_auc': auc,
                    'epoch': ep+1, 'labels': LABELS}, f'{CKPT_DIR}/model_best.pth')
        print(f'   saved best (mean AUC {best:.4f})')
print('best val mean AUC:', round(best,4))

## 5. Evaluate on the test set — per-class AUC + healthy verdict

Loads the best checkpoint, runs the held-out test set, and reports per-class AUC plus the derived **No Finding / Healthy** detector (`1 − max(probabilities)`).

In [ ]:
import matplotlib.pyplot as plt

ckpt = torch.load(f'{CKPT_DIR}/model_best.pth', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict']); model.eval()

test_loader = DataLoader(ChestXray(f'{OUT}/test.csv', False), batch_size=BATCH_SIZE*2, shuffle=False, **kw)
Y, P, mean_auc = evaluate(test_loader)
aucs, _ = class_auc(Y, P)

tbl = pd.DataFrame({'AUC': aucs}, index=LABELS).sort_values('AUC', ascending=False)
print(tbl.round(3).to_string()); print(f'\nmean AUC: {mean_auc:.4f}')

# derived healthy detector: abnormal score = max pathology prob
healthy_true = (Y.sum(axis=1) == 0).astype(int)          # 1 = healthy
abnormal_score = P.max(axis=1)                            # high = abnormal
print(f'"No Finding / Healthy" detector AUC: {roc_auc_score(1-healthy_true, abnormal_score):.4f}')

plt.figure(figsize=(8,5))
tbl['AUC'].plot.barh(color='steelblue'); plt.axvline(0.5, ls='--', c='gray')
plt.xlabel('Test AUC-ROC'); plt.title(f'Per-class AUC (mean {mean_auc:.3f})')
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

## 6. Demo — single prediction with a healthy/abnormal verdict

In [ ]:
def report(probs, threshold=0.5):
    findings = [(LABELS[i], probs[i]) for i in range(NUM_CLASSES) if probs[i] >= threshold]
    no_finding = 1 - probs.max()
    print(f'No Finding / Healthy score: {no_finding:.1%}')
    if findings:
        print('Verdict: ABNORMAL — flagged findings:')
        for name, p in sorted(findings, key=lambda x:-x[1]):
            print(f'   {name:20s} {p:.1%}')
    else:
        print('Verdict: likely HEALTHY (no pathology above threshold)')

# example: first test image
img, age, sex, y = ChestXray(f'{OUT}/test.csv', False)[0]
with torch.no_grad(), autocast():
    p = torch.sigmoid(model(img.unsqueeze(0).to(DEVICE), age.unsqueeze(0).to(DEVICE),
                            sex.unsqueeze(0).to(DEVICE)).float()).squeeze().cpu().numpy()
report(p)
print('\nground truth:', [LABELS[i] for i in range(NUM_CLASSES) if y[i]] or ['No Finding'])